### Data Preprocessing
* Builds manifest 
* Deterministic per-person 80/20 split into 80% enroll and 20% probe
* Creates sharded and merged embeddings and maps for each set


In [1]:
# ensure dependencies are installed in the current Python environment
import sys
!{sys.executable} -m pip install -q --upgrade jupyterlab pandas tqdm pillow numpy torch torchvision facenet-pytorch imagehash opencv-python-headless faiss-cpu scikit-learn matplotlib seaborn

In [2]:
# Config + imports
import os
from pathlib import Path
import hashlib
import json
from PIL import Image
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import imagehash
import cv2
import torch

# Paths: update these if needed
ROOT_DIR = Path("../data_raw")         # where your datasets live
OUT_DIR = Path("../data_processed")    # where processed outputs will go
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Run options
SAMPLE_LIMIT = None   # set to an int to process fewer images during testing
MIN_FACE_SIDE = 80    # min face side in pixels to accept (tweak for your data)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
repo_root = ROOT_DIR.parent.resolve()

def to_rel(p: Path) -> str:
    """Return a stable relative path string from repo_root, fallback to name."""
    try:
        return str(p.relative_to(repo_root).as_posix())
    except Exception:
        try:
            return str(p.relative_to(Path.cwd()).as_posix())
        except Exception:
            return str(p.name)

Device: cpu


In [4]:
# Check: counts the number of files on disk (in this case, images)

from pathlib import Path
img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".pgm"}   
vid_exts = {".mp4", ".avi", ".mov", ".mkv", ".wmv", ".flv", ".mpg", ".mpeg"}
exts = img_exts | vid_exts
ROOT = Path("../data_raw").resolve()   # same ROOT_DIR used; adjust if needed
print("Scanning disk under:", to_rel(ROOT))
if not ROOT.exists():
    raise FileNotFoundError(f"{ROOT} not found from this kernel. Check ROOT_DIR.")

files = [p for p in ROOT.rglob("*") if p.is_file() and p.suffix.lower() in exts]
print("Total media files on disk:", len(files))

from collections import Counter
tops = [p.relative_to(ROOT).parts[0] if len(p.relative_to(ROOT).parts) else "" for p in files]
for k,v in Counter(tops).most_common():
    print(f"  {k}: {v}")

Scanning disk under: data_raw
Total media files on disk: 198200
  vggface2: 197693
  ibeta1: 507


In [6]:
# Manifest builder (per-dataset + master)
# - Scans ROOT_DIR for image files
# - Computes sha1, perceptual hash, image size, readable flag
# - Writes per-dataset manifest: data_processed/<dataset>/manifests/manifest_basic.csv
# - Writes master manifest: data_processed/manifests/master_manifest_basic.csv

import sys
from pathlib import Path
import hashlib
from PIL import Image, UnidentifiedImageError
import imagehash
import pandas as pd
from tqdm.auto import tqdm

# Config (adjust as needed)
ROOT_DIR = Path("../data_raw").resolve()        # where raw datasets live
OUT_ROOT = Path("../data_processed").resolve()  # output location for processed outputs
SAMPLE_LIMIT = None                             # set to an int for quick tests
exts = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".pgm", ".webp"}

repo_root = ROOT_DIR.parent.resolve()

OUT_ROOT.mkdir(parents=True, exist_ok=True)
(OUT_ROOT / "manifests").mkdir(parents=True, exist_ok=True)

# used for detection of duplicates (i.e. bytewise identical files)
def sha1_of_file(path: Path, block_size: int = 65536):
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(block_size), b""):
            h.update(b)
    return h.hexdigest()

# used for detection of near-duplicates (i.e. same image resized/recompressed)
def phash_of_image(path: Path):
    try:
        return str(imagehash.average_hash(Image.open(path)))
    except Exception:
        return None

if not ROOT_DIR.exists():
    raise FileNotFoundError(f"ROOT_DIR not found: {ROOT_DIR}")

# Collect files (respect SAMPLE_LIMIT if set)
all_files = []
count = 0
for p in ROOT_DIR.rglob("*"):
    if not p.is_file():
        continue
    if p.suffix.lower() not in exts:
        continue
    all_files.append(p)
    count += 1
    if SAMPLE_LIMIT and count >= SAMPLE_LIMIT:
        break

print(f"Found {len(all_files)} image files under {to_rel(ROOT_DIR)}")

# Build master rows
master_rows = []

# Build per-dataset lists to write separate CSVs
per_dataset_rows = {}

for p in tqdm(sorted(all_files), desc="Scanning images"):
    try:
        rel = p.relative_to(ROOT_DIR)
    except Exception:
        # unexpected path; fallback to name only
        rel = Path(p.name)
    parts = rel.parts
    dataset = parts[0] if len(parts) > 0 else ""
    split = parts[1] if len(parts) > 1 else ""
    person_raw = parts[2] if len(parts) > 2 else p.parent.name
    person_id = person_raw   

    # Ensure per-dataset output directories exist
    ds_out = OUT_ROOT / dataset
    (ds_out / "aligned").mkdir(parents=True, exist_ok=True)
    (ds_out / "embeddings").mkdir(parents=True, exist_ok=True)
    (ds_out / "manifests").mkdir(parents=True, exist_ok=True)

    # Read image metadata
    w = h = fmt = None
    readable = False
    try:
        with Image.open(p) as im:
            w, h = im.size
            fmt = im.format
            readable = True
    except (UnidentifiedImageError, OSError, ValueError):
        readable = False
    except Exception:
        readable = False

    sha1 = None
    phash = None
    try:
        sha1 = sha1_of_file(p)
    except Exception:
        sha1 = None
    try:
        phash = phash_of_image(p)
    except Exception:
        phash = None

    row = {
        "image_path": str(p),           # path to original image
        "dataset": dataset,             # source labels (e.g. vggface2,ibeta)
        "split": split,                 # split (e.g. train, val, test)
        "person_id": person_id,         # person id (e.g. n000002)
        "person_raw": person_raw,       # raw person id extracted from file, so person_id can be renamed/reconstructed
        "sha1": sha1,                   # sha1 hash of file (checksum that is used to detect duplicates)
        "phash": phash,                 # perceptual hash of image (for near-duplicate detection)
        "width": w,                     # image width in pixels
        "height": h,                    # image height in pixels
        "format": fmt,                  # image format (e.g. JPEG, PNG)
        "readable": bool(readable)      # whether image could be opened/read
    }

    master_rows.append(row)
    per_dataset_rows.setdefault(dataset, []).append(row)

# Write master manifest
master_df = pd.DataFrame(master_rows)
master_path = OUT_ROOT / "manifests" / "master_manifest_basic.csv"
master_df.to_csv(master_path, index=False)
print(f"Wrote master manifest: {to_rel(master_path)} rows: {len(master_df)}")

# Write per-dataset manifests
for dataset, rows in per_dataset_rows.items():
    ds_manifest_path = OUT_ROOT / dataset / "manifests" / "manifest_basic.csv"
    pd.DataFrame(rows).to_csv(ds_manifest_path, index=False)
    print(f"Wrote {dataset} manifest: {to_rel(ds_manifest_path)} rows: {len(rows)}")

print("Manifest building complete.")

Found 197693 image files under data_raw


Scanning images: 100%|██████████| 197693/197693 [40:58<00:00, 80.41it/s] 


Wrote master manifest: data_processed/manifests/master_manifest_basic.csv rows: 197693
Wrote vggface2 manifest: data_processed/vggface2/manifests/manifest_basic.csv rows: 197693
Manifest building complete.


In [49]:
# Create manifest_enroll.csv for 80/20 enroll/probe split 
# - Existing manifest_basic.csv contains all images with person_id labels
# - For each person_id, randomly select 80% of images for 'enroll'
# - The remaining 20% of images are used for 'probe'
# - Writes manifest_enroll.csv and manifest_probe.csv
# - This will be used for identity evaluation later

import pandas as pd
import numpy as np
from pathlib import Path
import math
import hashlib

DS_MANIFEST = Path("../data_processed/vggface2/manifests/manifest_basic.csv")
POOL_TRAIN_VAL = True        # Pool both 'train' and 'val' rows before splitting
ENROLL_FRAC = 0.8            # enroll 80% of images per person
GLOBAL_SEED = 42             # deterministic seed
CAP_ENROLL = None            # set to int (e.g. 20) to cap #enroll images per person, or None

# Read manifest
df = pd.read_csv(DS_MANIFEST)
print(f"Read manifest: {len(df)} rows from {DS_MANIFEST}")

# Filter to train+val (or just train) before grouping
if POOL_TRAIN_VAL:
    df = df[df['split'].isin(['train', 'val'])].reset_index(drop=True)
else:
    df = df[df['split'] == 'train'].reset_index(drop=True)
print(f"Using {len(df)} rows after filtering splits (pool_train_val={POOL_TRAIN_VAL})")

# Prepare containers
enroll_list = []
probe_list = []

# Group by person_id
for person_id, g in df.groupby('person_id', sort=True):
    g = g.reset_index(drop=True)
    n = len(g)
    # deterministic RNG per person (mix global seed + person hash)
    # use hashlib to get consistent hash across runs
    seed = (GLOBAL_SEED + int(hashlib.sha1(person_id.encode('utf8')).hexdigest()[:8], 16)) & 0xFFFFFFFF
    rng = np.random.RandomState(seed)
    perm = rng.permutation(n)
    # determine number of enroll images (ensure at least 1 if n>=1)
    if n == 1:
        n_enroll = 1
    else:
        n_enroll = max(1, int(math.floor(n * ENROLL_FRAC)))
    if CAP_ENROLL is not None:
        n_enroll = min(n_enroll, CAP_ENROLL)
    enroll_idx = perm[:n_enroll]
    probe_idx  = perm[n_enroll:]
    enroll_list.append(g.iloc[enroll_idx])
    if len(probe_idx) > 0:
        probe_list.append(g.iloc[probe_idx])

# Concat results
enroll_df = pd.concat(enroll_list, ignore_index=True) if enroll_list else pd.DataFrame(columns=df.columns)
probe_df  = pd.concat(probe_list,  ignore_index=True) if probe_list  else pd.DataFrame(columns=df.columns)

out_dir = Path("../data_processed/vggface2/manifests")
out_dir.mkdir(parents=True, exist_ok=True)
enroll_path = out_dir / "manifest_enroll.csv"
probe_path  = out_dir / "manifest_probe.csv"

enroll_df.to_csv(enroll_path, index=False)
probe_df.to_csv(probe_path, index=False)

print(f"Wrote enroll manifest: {enroll_path}  rows: {len(enroll_df)}  unique persons: {enroll_df['person_id'].nunique()}")
print(f"Wrote probe  manifest: {probe_path}   rows: {len(probe_df)}  unique persons: {probe_df['person_id'].nunique()}")

# Diagnostics
single_image_persons = (df.groupby('person_id').size() == 1).sum()
persons_no_probe = enroll_df['person_id'].nunique() - probe_df['person_id'].nunique()
print(f"Persons with only one image in original data: {single_image_persons}")
print(f"Persons present in enroll but not in probe: {persons_no_probe}")


Read manifest: 197693 rows from ..\data_processed\vggface2\manifests\manifest_basic.csv
Using 197693 rows after filtering splits (pool_train_val=True)
Wrote enroll manifest: ..\data_processed\vggface2\manifests\manifest_enroll.csv  rows: 157935  unique persons: 540
Wrote probe  manifest: ..\data_processed\vggface2\manifests\manifest_probe.csv   rows: 39758  unique persons: 540
Persons with only one image in original data: 0
Persons present in enroll but not in probe: 0


In [50]:
# Split manifest into train/val files 

src = "../data_processed/vggface2/manifests/manifest_basic.csv"
df = pd.read_csv(src)
df_train = df[df["split"] == "train"].reset_index(drop=True)
df_val   = df[df["split"] == "val"].reset_index(drop=True)
df_train.to_csv("../data_processed/vggface2/manifests/manifest_train.csv", index=False)
df_val.to_csv("../data_processed/vggface2/manifests/manifest_val.csv", index=False)
print(len(df_train), "train rows;", len(df_val), "val rows")

176398 train rows; 21295 val rows


In [ ]:
# Run embedding extraction + merge for train/val and enroll/probe
# - Purpose: extract face embeddings using facenet-pytorch InceptionResnetV1 model
# - Uses scripts/embeddings.py module
# - Embeddings are stored as sharded files and then merged for easier downstream use
# - Merged files can be used for training/evaluation later
# - Outputs are stored under ../data_processed/vggface2/embeddings/

from pathlib import Path
import sys, importlib

# Paths & params
OUT_TRAIN  = "../data_processed/vggface2/embeddings/train"
OUT_VAL    = "../data_processed/vggface2/embeddings/val"
OUT_ENROLL = "../data_processed/vggface2/embeddings/enroll"
OUT_PROBE  = "../data_processed/vggface2/embeddings/probe"

MAN_TRAIN  = "../data_processed/vggface2/manifests/manifest_train.csv"
MAN_VAL    = "../data_processed/vggface2/manifests/manifest_val.csv"
MAN_ENROLL = "../data_processed/vggface2/manifests/manifest_enroll.csv"
MAN_PROBE  = "../data_processed/vggface2/manifests/manifest_probe.csv"

BATCH_SIZE = 128
SHARD_SIZE = 2000
SAMPLE_LIMIT = None  # set to int for quick tests

# Ensure notebook can import the `scripts/embeddings.py` module
scripts_dir = Path.cwd() / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.append(str(scripts_dir))

import embeddings as embeddings_module
importlib.reload(embeddings_module)

# Helper to run embedding + merge + status 
def run_and_merge(manifest_path, out_dir, batch_size=BATCH_SIZE, shard_size=SHARD_SIZE, sample_limit=SAMPLE_LIMIT):
    print(f"=== Processing manifest: {manifest_path} -> {out_dir} ===")
    embeddings_module.embed_sharded(manifest_path, out_dir, batch_size=batch_size, shard_size=shard_size, sample_limit=sample_limit)
    embeddings_module.merge_shards(out_dir)
    embeddings_module.status(out_dir)
    print("")

# Run for train/val 
if Path(MAN_TRAIN).exists():
    run_and_merge(MAN_TRAIN, OUT_TRAIN)
else:
    print("Skipping train: manifest not found:", MAN_TRAIN)

if Path(MAN_VAL).exists():
    run_and_merge(MAN_VAL, OUT_VAL)
else:
    print("Skipping val: manifest not found:", MAN_VAL)

# Run for enroll/probe (new closed-set split)
if Path(MAN_ENROLL).exists():
    run_and_merge(MAN_ENROLL, OUT_ENROLL)
else:
    print("Skipping enroll: manifest not found:", MAN_ENROLL)

if Path(MAN_PROBE).exists():
    run_and_merge(MAN_PROBE, OUT_PROBE)
else:
    print("Skipping probe: manifest not found:", MAN_PROBE)

print("All done. Check outputs under ../data_processed/vggface2/embeddings/")

=== Processing manifest: ../data_processed/vggface2/manifests/manifest_train.csv -> ../data_processed/vggface2/embeddings/train ===


2025-11-26 04:17:39,659 INFO No new rows to process.
2025-11-26 04:17:42,958 INFO Merged embeddings -> ..\data_processed\vggface2\embeddings\train\embeddings.npy (rows=176398)
2025-11-26 04:17:42,960 INFO Merged map -> ..\data_processed\vggface2\embeddings\train\embeddings_map.csv (rows=176398)


out_dir: ..\data_processed\vggface2\embeddings\train
shard embeddings: 87
shard maps: 87
final embeddings exists: True
final map exists: True

=== Processing manifest: ../data_processed/vggface2/manifests/manifest_val.csv -> ../data_processed/vggface2/embeddings/val ===


2025-11-26 04:17:43,519 INFO No new rows to process.
2025-11-26 04:17:43,888 INFO Merged embeddings -> ..\data_processed\vggface2\embeddings\val\embeddings.npy (rows=21295)
2025-11-26 04:17:43,888 INFO Merged map -> ..\data_processed\vggface2\embeddings\val\embeddings_map.csv (rows=21295)


out_dir: ..\data_processed\vggface2\embeddings\val
shard embeddings: 11
shard maps: 11
final embeddings exists: True
final map exists: True

=== Processing manifest: ../data_processed/vggface2/manifests/manifest_enroll.csv -> ../data_processed/vggface2/embeddings/enroll ===


rows: 100%|██████████| 157935/157935 [1:02:32<00:00, 42.09it/s]
2025-11-26 05:20:19,272 INFO Wrote shard embeddings_shard_0077.npy (rows=239)
2025-11-26 05:20:19,272 INFO Total new embeddings produced: 157935
2025-11-26 05:20:24,018 INFO Merged embeddings -> ..\data_processed\vggface2\embeddings\enroll\embeddings.npy (rows=157935)
2025-11-26 05:20:24,018 INFO Merged map -> ..\data_processed\vggface2\embeddings\enroll\embeddings_map.csv (rows=157935)


out_dir: ..\data_processed\vggface2\embeddings\enroll
shard embeddings: 78
shard maps: 78
final embeddings exists: True
final map exists: True

=== Processing manifest: ../data_processed/vggface2/manifests/manifest_probe.csv -> ../data_processed/vggface2/embeddings/probe ===


rows: 100%|██████████| 39758/39758 [15:47<00:00, 41.94it/s]
2025-11-26 05:36:13,607 INFO Wrote shard embeddings_shard_0019.npy (rows=846)
2025-11-26 05:36:13,608 INFO Total new embeddings produced: 39758
2025-11-26 05:36:14,751 INFO Merged embeddings -> ..\data_processed\vggface2\embeddings\probe\embeddings.npy (rows=39758)
2025-11-26 05:36:14,751 INFO Merged map -> ..\data_processed\vggface2\embeddings\probe\embeddings_map.csv (rows=39758)


out_dir: ..\data_processed\vggface2\embeddings\probe
shard embeddings: 20
shard maps: 20
final embeddings exists: True
final map exists: True

All done. Check outputs under ../data_processed/vggface2/embeddings/


In [40]:
# POSE + PAD annotation (with alignment checks & checkpointing)
import time
import os
import torch
from pathlib import Path
import pandas as pd
from facenet_pytorch import MTCNN
from sklearn.cluster import KMeans
from PIL import Image
import cv2
import numpy as np
import math
from tqdm.auto import tqdm

# local helpers
from scripts.pose import landmarks_to_pose
from scripts.pad import heuristic_liveness_score

OUT_DIR = Path("../data_processed/vggface2")
SPLIT = "enroll"   # change to "probe" / "train" as needed
MAP_PATH = OUT_DIR / "embeddings" / SPLIT / "embeddings_map.csv"
IMG_ROOT = None   # optional prefix
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Progress + heuristics
LOG_EVERY = 5000            # print a short line every LOG_EVERY images
MIN_FACE_SIDE = 80          # min face side (px) for aligned face
MIN_CONF = 0.9              # min face detection confidence for aligned face
CHECKPOINT_EVERY = 1000     # write a checkpoint CSV every N images

# checkpoint directory (will be created if missing)
CHECKPOINT_DIR = OUT_DIR / "embeddings" / SPLIT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Annotating split='{SPLIT}' device={DEVICE}")

# load map and paths
df = pd.read_csv(MAP_PATH)
if IMG_ROOT:
    paths = [str(Path(IMG_ROOT) / p) for p in df['image_path'].astype(str).tolist()]
else:
    paths = df['image_path'].astype(str).tolist()
total = len(paths)
print(f"  Images to process: {total}")

# setup MTCNN
mtcnn = MTCNN(keep_all=False, device=DEVICE)

# output containers
yaws, pitches, rolls = [], [], []
landmark_list = []
liveness_scores = []
aligned_flags = []
box_areas = []

# iterate once: compute landmarks -> pose and compute heuristic liveness from loaded image
for i, p in enumerate(tqdm(paths, desc="Annotating", total=total), start=1):
    try:
        img = Image.open(p).convert("RGB")
        w, h = img.size
        box, probs, landmarks = mtcnn.detect(img, landmarks=True)
        if landmarks is None or len(landmarks) == 0:
            yaws.append(float("nan")); pitches.append(float("nan")); rolls.append(float("nan"))
            landmark_list.append(None)
            aligned_flags.append(False)
            box_areas.append(0.0)
        else:
            lm = landmarks[0]  # (5,2)
            yaw, pitch, roll = landmarks_to_pose(lm, (w, h))
            yaws.append(yaw); pitches.append(pitch); rolls.append(roll)
            landmark_list.append(lm.tolist())

            # compute bounding box area & basic aligned-flag
            if box is not None and len(box) > 0:
                b = box[0]
                bw = max(0.0, float(b[2] - b[0])); bh = max(0.0, float(b[3] - b[1]))
                box_area = bw * bh
                box_areas.append(box_area)
                conf = float(probs[0]) if (probs is not None and len(probs) > 0) else 0.0
                aligned = (min(bw, bh) >= MIN_FACE_SIDE) and (conf >= MIN_CONF)
                aligned_flags.append(bool(aligned))
            else:
                box_areas.append(0.0)
                aligned_flags.append(False)

        # compute liveness from the same loaded image (avoid second file read)
        bgr = cv2.cvtColor(np.asarray(img), cv2.COLOR_RGB2BGR)
        live = heuristic_liveness_score(bgr)
        liveness_scores.append(float(live))
    except Exception:
        yaws.append(float("nan")); pitches.append(float("nan")); rolls.append(float("nan"))
        landmark_list.append(None)
        liveness_scores.append(float("nan"))
        aligned_flags.append(False)
        box_areas.append(0.0)

    # periodic lightweight logging
    if (i % LOG_EVERY) == 0:
        print(f"  processed {i}/{total} images (last path: {Path(p).name})")

    # checkpoint: write partial CSV so long runs can be resumed/inspected
    if (i % CHECKPOINT_EVERY) == 0:
        ck = pd.DataFrame({
            "image_path": paths[:i],
            "yaw": yaws,
            "pitch": pitches,
            "roll": rolls,
            "pose_landmarks": landmark_list,
            "liveness_score": liveness_scores,
            "face_aligned": aligned_flags,
            "face_box_area": box_areas,
        })
        ts = time.strftime("%Y%m%dT%H%M%S")
        fname = f"embeddings_map_with_pose_liveness_checkpoint_{i}_{ts}.csv"
        tmpf = CHECKPOINT_DIR / (fname + ".tmp")
        finalf = CHECKPOINT_DIR / fname
        # atomic-ish write
        ck.to_csv(tmpf, index=False)
        os.replace(str(tmpf), str(finalf))

# attach columns to full dataframe
df['yaw'] = yaws
df['pitch'] = pitches
df['roll'] = rolls
df['pose_landmarks'] = landmark_list
df['liveness_score'] = liveness_scores
df['face_aligned'] = aligned_flags
df['face_box_area'] = box_areas

# CLUSTER: KMeans on yaw (K=3) or use deterministic bins
USE_KMEANS = True
K = 3
if USE_KMEANS:
    X = df['yaw'].fillna(0.0).to_numpy().reshape(-1,1).astype(float)
    k = min(K, max(1, X.shape[0]))
    if k == 1:
        df['pose_cluster'] = 0
    else:
        km = KMeans(n_clusters=k, random_state=12345)
        labels = km.fit_predict(X)
        df['pose_cluster'] = labels.astype(int)
else:
    bins = [-999, -15.0, 15.0, 999]
    labels = pd.cut(df['yaw'].fillna(0.0), bins=bins, labels=False)
    df['pose_cluster'] = labels.fillna(1).astype(int)

# write augmented map
OUT_MAP = MAP_PATH.with_name(MAP_PATH.stem + "_with_pose_liveness.csv")
df.to_csv(OUT_MAP, index=False)

# final minimal summaries
print(f"Wrote: {OUT_MAP}")
print("  pose_cluster counts:", df['pose_cluster'].value_counts(dropna=False).to_dict())
print("  liveness (mean,std):", float(df['liveness_score'].mean(skipna=True)), float(df['liveness_score'].std(skipna=True)))

Annotating split='enroll' device=cpu
  Images to process: 157935


Annotating:   3%|▎         | 5000/157935 [04:02<3:03:03, 13.92it/s]

  processed 5000/157935 images (last path: 0098_01.jpg)


Annotating:   6%|▋         | 9999/157935 [08:39<2:14:05, 18.39it/s]

  processed 10000/157935 images (last path: 0022_01.jpg)


...

  processed 150000/157935 images (last path: 0137_03.jpg)


Annotating:  98%|█████████▊| 154998/157935 [2:29:48<03:38, 13.44it/s]  

  processed 155000/157935 images (last path: 0302_01.jpg)


Annotating: 100%|██████████| 157935/157935 [2:33:05<00:00, 17.19it/s]


Wrote: ..\data_processed\vggface2\embeddings\enroll\embeddings_map_with_pose_liveness.csv
  pose_cluster counts: {0: 86202, 1: 36580, 2: 35153}
  liveness (mean,std): 0.5820900719786825 0.08974859842720338


In [41]:
import pandas as pd
df = pd.read_csv(r"../data_processed/vggface2/embeddings/enroll/embeddings_map_with_pose_liveness.csv")
# cluster -> mean yaw (helps assign left/frontal/right)
centers = df.groupby('pose_cluster')['yaw'].agg(['mean','std','count']).sort_values('mean')
print("Cluster yaw summary:\n", centers)
# per-cluster liveness
print("Per-cluster liveness:\n", df.groupby('pose_cluster')['liveness_score'].agg(['mean','std','count']))

order = centers.reset_index().sort_values('mean')['pose_cluster'].tolist()
label_map = {}
labels = ['left','frontal','right'] if len(order)==3 else [f'c{i}' for i in order]
for lbl, cid in zip(labels, order):
    label_map[cid] = lbl
print("Label map:", label_map)
df['pose_label'] = df['pose_cluster'].map(label_map)

print("\nOverall liveness mean,std:", df['liveness_score'].mean(), df['liveness_score'].std())
for cid, g in df.groupby('pose_cluster'):
    print(f"Cluster {cid} ({label_map.get(cid,cid)}): n={len(g)}, liveness_mean={g['liveness_score'].mean():.3f}, below0.5={ (g['liveness_score']<0.5).mean():.3f}")

print("\n")
# compute mean liveness per person (and optionally save)
person_live = df.groupby('person_id')['liveness_score'].agg(['mean','count']).reset_index().rename(columns={'mean':'mean_liveness','count':'n_images'})
person_live.to_csv("../data_processed/vggface2/person_mean_liveness_enroll.csv", index=False)
print(person_live.describe())

Cluster yaw summary:
                    mean        std  count
pose_cluster                             
2            -35.219052  15.009354  35153
0             -0.410645   9.420023  85947
1             34.292154  15.266236  36580
Per-cluster liveness:
                   mean       std  count
pose_cluster                           
0             0.581397  0.090897  86202
1             0.583091  0.087873  36580
2             0.582748  0.088828  35153
Label map: {2: 'left', 0: 'frontal', 1: 'right'}

Overall liveness mean,std: 0.5820900719786825 0.08974859842720338
Cluster 0 (frontal): n=86202, liveness_mean=0.581, below0.5=0.125
Cluster 1 (right): n=36580, liveness_mean=0.583, below0.5=0.118
Cluster 2 (left): n=35153, liveness_mean=0.583, below0.5=0.118


       mean_liveness    n_images
count     540.000000  540.000000
mean        0.582601  292.472222
std         0.019493   84.820520
min         0.525073   81.000000
25%         0.570819  229.500000
50%         0.581701  292.500000
75%

In [42]:
import pandas as pd
from pathlib import Path

p = Path("../data_processed/vggface2/embeddings/enroll/embeddings_map_with_pose_liveness.csv")
df = pd.read_csv(p)
print("rows:", len(df))
print("columns:", df.columns.tolist())
print("pose_cluster unique:", df.get('pose_cluster').unique() if 'pose_cluster' in df.columns else 'no column')
print("yaw NaNs:", df['yaw'].isna().sum(), "non-NaN:", df['yaw'].notna().sum())
print("yaw describe:\n", df['yaw'].describe())
# show a few yaw values
print(df['yaw'].dropna().head(20).to_list()[:20])

rows: 157935
columns: ['embedding_index', 'embedding_index.1', 'image_path', 'dataset', 'person_id', 'sha1', 'yaw', 'pitch', 'roll', 'pose_landmarks', 'liveness_score', 'face_aligned', 'face_box_area', 'pose_cluster']
pose_cluster unique: [0 1 2]
yaw NaNs: 255 non-NaN: 157680
yaw describe:
 count    157680.000000
mean         -0.120130
std          26.496681
min         -89.441580
25%         -15.639062
50%          -0.102677
75%          15.377024
max          89.079609
Name: yaw, dtype: float64
[-16.315277300229816, -11.13502263904917, -15.21562776404591, 6.449727803275582, 0.7791751447296063, -3.667393770113618, -11.4105325834311, 59.82140655472836, 13.77451428937662, -47.64492967940741, -19.89324308652697, -42.76427953267803, 72.72865387728912, 36.19324225913604, -6.541681934400658, -13.490957380789249, -6.857723182568478, 8.771760986463747, 5.257308703840348, 1.169998842172308]


In [43]:
# Quick simulation: Simulate candidate K vs MIN_IMAGES thresholds
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from pathlib import Path

p = Path("../data_processed/vggface2/embeddings/enroll/embeddings_map_with_pose_liveness.csv")
df = pd.read_csv(p)

valid = df['yaw'].notna()
yaw = df.loc[valid, 'yaw'].to_numpy().reshape(-1,1).astype(float)

candidate_K = [2, 3, 4, 5, 7]
min_images_list = [1, 2, 3, 5]

rows = []
for K in candidate_K:
    # cluster only valid rows
    if yaw.shape[0] == 0:
        labels_full = np.full(len(df), -1, dtype=int)  # all unknown
    else:
        k = min(K, max(1, yaw.shape[0]))
        if k == 1:
            labels = np.zeros(yaw.shape[0], dtype=int)
        else:
            km = KMeans(n_clusters=k, random_state=12345)
            labels = km.fit_predict(yaw)

        # map labels back into full-length array; invalid yaw -> -1
        labels_full = np.full(len(df), -1, dtype=int)
        labels_full[valid.to_numpy()] = labels

    tmp = df.copy()
    tmp['sim_label'] = labels_full

    # compute centers ignoring unknowns
    valid_tmp = tmp[tmp['sim_label'] >= 0]
    if valid_tmp.shape[0] == 0:
        centers = pd.Series(dtype=float)
    else:
        centers = valid_tmp.groupby('sim_label')['yaw'].mean().sort_values()

    order = centers.index.tolist()
    mapping = {old: new for new, old in enumerate(order)}
    # remap only valid labels
    tmp.loc[tmp['sim_label'] >= 0, 'sim_label'] = tmp.loc[tmp['sim_label'] >= 0, 'sim_label'].map(mapping).astype(int)

    # grouped counts only for valid labels
    grouped = tmp[tmp['sim_label'] >= 0].groupby(['person_id', 'sim_label']).size().reset_index(name='count')

    for min_images in min_images_list:
        n_templates = int((grouped['count'] >= min_images).sum())
        # use agg on the Series to avoid the FutureWarning
        avg_tpls_per_person = grouped.groupby('person_id')['count'].agg(lambda s: (s >= min_images).sum()).mean()
        rows.append({
            'K': K,
            'min_images': min_images,
            'n_templates': n_templates,
            'avg_templates_per_person': float(avg_tpls_per_person),
            'frac_persons_with_>=3_poses': float((tmp[tmp['sim_label'] >= 0].groupby('person_id')['sim_label'].nunique() >= 3).mean())
        })

res = pd.DataFrame(rows)
print("Templates by K / min_images:\n", res.pivot(index='K', columns='min_images', values='n_templates'))
print("\nAvg templates per person (sample):\n", res.pivot(index='K', columns='min_images', values='avg_templates_per_person'))
print("\nFraction persons with >=3 pose clusters (same for all min_images):")
print(res[['K','frac_persons_with_>=3_poses']].drop_duplicates().set_index('K'))

# per-person coverage histogram for this K
hist = tmp[tmp['yaw'].notna()].groupby('person_id')['sim_label'].nunique().value_counts().sort_index()
print(f"K={K} per-person pose-coverage (num persons -> count):\n", hist)


Templates by K / min_images:
 min_images     1     2     3     5
K                                 
2           1080  1080  1080  1080
3           1620  1620  1620  1620
4           2160  2160  2160  2160
5           2700  2700  2700  2700
7           3779  3777  3769  3724

Avg templates per person (sample):
 min_images         1         2        3         5
K                                                
2           2.000000  2.000000  2.00000  2.000000
3           3.000000  3.000000  3.00000  3.000000
4           4.000000  4.000000  4.00000  4.000000
5           5.000000  5.000000  5.00000  5.000000
7           6.998148  6.994444  6.97963  6.896296

Fraction persons with >=3 pose clusters (same for all min_images):
   frac_persons_with_>=3_poses
K                             
2                          0.0
3                          1.0
4                          1.0
5                          1.0
7                          1.0
K=7 per-person pose-coverage (num persons -> count):


In [44]:
# Previous cell showed K=5 gives good coverage
# Recluster to K=5 and save map
# Reclustering yaw -> K=5 (canonical left->...->right) and save map
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from pathlib import Path

MAP_IN = Path("../data_processed/vggface2/embeddings/enroll/embeddings_map_with_pose_liveness.csv")
MAP_OUT = MAP_IN.with_name(MAP_IN.stem + "_k5.csv")
df = pd.read_csv(MAP_IN)

# Use yaw for clustering; fill NaNs with 0 but keep count
X = df['yaw'].fillna(0.0).to_numpy().reshape(-1,1).astype(float)
K = 5
km = KMeans(n_clusters=K, random_state=12345)
labels = km.fit_predict(X)
df['pose_cluster_k5_raw'] = labels

# canonicalize cluster ids by increasing yaw center (so ids map left->right)
centers = df.groupby('pose_cluster_k5_raw')['yaw'].mean().sort_values()
order = centers.index.tolist()
mapping = {old:new for new, old in enumerate(order)}
df['pose_cluster_k5'] = df['pose_cluster_k5_raw'].map(mapping).astype(int)

# Add readable label names (0..4 -> left->...->right)
df['pose_label_k5'] = df['pose_cluster_k5'].map({i: f'pose_{i}' for i in range(K)})

# drop helper raw column
df = df.drop(columns=['pose_cluster_k5_raw'])

# drop duplicate index column if present
if 'embedding_index.1' in df.columns:
    df = df.drop(columns=['embedding_index.1'])

df.to_csv(MAP_OUT, index=False)
print("Wrote reclustered map:", MAP_OUT)
print("pose counts (k5):\n", df['pose_cluster_k5'].value_counts().sort_index())
print("yaw centers (k5):\n", df.groupby('pose_cluster_k5')['yaw'].mean().sort_index())

Wrote reclustered map: ..\data_processed\vggface2\embeddings\enroll\embeddings_map_with_pose_liveness_k5.csv
pose counts (k5):
 pose_cluster_k5
0    12288
1    36070
2    56613
3    39145
4    13819
Name: count, dtype: int64
yaw centers (k5):
 pose_cluster_k5
0   -52.163827
1   -21.885969
2    -0.967298
3    19.688269
4    50.314177
Name: yaw, dtype: float64


In [45]:
import pandas as pd
p = "../data_processed/vggface2/embeddings/enroll/embeddings_map_with_pose_liveness_k5.csv"
print("Columns:", pd.read_csv(p, nrows=2).columns.tolist())
print(pd.read_csv(p, nrows=5))

cols = ['embedding_index','person_id','yaw','pose_cluster_k5','pose_label_k5','liveness_score','face_aligned']
print(pd.read_csv(p, usecols=cols, nrows=20))

req = ['embedding_index','person_id','yaw','liveness_score','face_aligned','pose_cluster_k5','pose_label_k5','face_box_area']
head = pd.read_csv(p, nrows=1)
missing = [c for c in req if c not in head.columns.tolist()]
print("Missing columns:", missing)
# read only necessary columns (fast)
df = pd.read_csv(p, usecols=req)
print("rows:", len(df))
print("missing yaw:", df['yaw'].isna().sum())
# treat pose_cluster_k5 numeric; detect sentinel -1 or NaN
print("pose_cluster_k5 value counts (top):")
print(df['pose_cluster_k5'].value_counts(dropna=False).head(10))
print("face_aligned counts:", df['face_aligned'].value_counts(dropna=False))

Columns: ['embedding_index', 'image_path', 'dataset', 'person_id', 'sha1', 'yaw', 'pitch', 'roll', 'pose_landmarks', 'liveness_score', 'face_aligned', 'face_box_area', 'pose_cluster', 'pose_cluster_k5', 'pose_label_k5']
   embedding_index                                         image_path  \
0                0  C:\Users\yukki\Documents\CS228-Biometric-with-...   
1                1  C:\Users\yukki\Documents\CS228-Biometric-with-...   
2                2  C:\Users\yukki\Documents\CS228-Biometric-with-...   
3                3  C:\Users\yukki\Documents\CS228-Biometric-with-...   
4                4  C:\Users\yukki\Documents\CS228-Biometric-with-...   

    dataset person_id                                      sha1        yaw  \
0  vggface2   n000001  f356106348a50cf9828c108564e46fc1a6f74595        NaN   
1  vggface2   n000001  85e7ac5d644006dbf5565fec4579e43171d87efa -16.315277   
2  vggface2   n000001  cdb422b47fd6b4b46484a85dbb30e29e13b8bcb5 -11.135023   
3  vggface2   n000001  51b109

In [46]:
import pandas as pd
from pathlib import Path
p = Path("../data_processed/vggface2/embeddings/enroll/embeddings_map_with_pose_liveness_k5.csv")
df = pd.read_csv(p, usecols=['embedding_index','image_path','yaw','pose_landmarks','liveness_score','face_aligned','face_box_area'])

total = len(df)
na_landmarks = df['pose_landmarks'].isna().sum()
small_box = (df['face_box_area'] <= 0).sum()   # area==0 indicates missing box
# also compute inferred min-side threshold (approx) if you stored bw/bh; else estimate sqrt(area)
min_side_est = (df['face_box_area']**0.5)  # rough estimate; compare with MIN_FACE_SIDE
small_side_count = (min_side_est < 80).sum()  # same threshold as notebook default
print("total:", total)
print("missing landmarks:", na_landmarks)
print("zero box area (likely missing):", small_box)
print("estimated min-side < 80 (approx):", small_side_count)
print("face_aligned True:", df['face_aligned'].sum(), "False:", total - df['face_aligned'].sum())

# distribution of yaw for aligned vs not-aligned
print("yaw describe for aligned rows:")
print(df[df['face_aligned']==True]['yaw'].describe())
print("yaw describe for not-aligned rows:")
print(df[df['face_aligned']==False]['yaw'].describe())

# liveness distribution vs alignment
print("mean liveness aligned:", df[df['face_aligned']==True]['liveness_score'].mean())
print("mean liveness not-aligned:", df[df['face_aligned']==False]['liveness_score'].mean())



total: 157935
missing landmarks: 255
zero box area (likely missing): 255
estimated min-side < 80 (approx): 56998
face_aligned True: 86189 False: 71746
yaw describe for aligned rows:
count    86189.000000
mean         0.064061
std         26.169992
min        -89.279653
25%        -14.911423
50%          0.077920
75%         14.941327
max         88.900831
Name: yaw, dtype: float64
yaw describe for not-aligned rows:
count    71491.000000
mean        -0.342189
std         26.883763
min        -89.441580
25%        -16.533100
50%         -0.333879
75%         15.865128
max         89.079609
Name: yaw, dtype: float64
mean liveness aligned: 0.5646128758768142
mean liveness not-aligned: 0.6030855568116198


In [47]:
import pandas as pd
p = "../data_processed/vggface2/embeddings/enroll/embeddings_map_with_pose_liveness_k5.csv"
df = pd.read_csv(p, usecols=['embedding_index','face_box_area','face_aligned','liveness_score','pose_landmarks'])

# approximate min-side from area
df['min_side_est'] = (df['face_box_area']**0.5)
print("min_side_est describe:\n", df['min_side_est'].describe())

# correlation between box area and liveness
print("corr(box_area, liveness):", df['face_box_area'].corr(df['liveness_score']))

# fraction of images below thresholds
print("frac min-side <80:", (df['min_side_est'] < 80).mean())
print("face_aligned True fraction:", df['face_aligned'].mean())

min_side_est describe:
 count    157935.000000
mean        122.626045
std          81.984504
min           0.000000
25%          66.004842
50%          99.734741
75%         156.228120
max        2017.458178
Name: min_side_est, dtype: float64
corr(box_area, liveness): -0.2187028198893316
frac min-side <80: 0.36089530503055056
face_aligned True fraction: 0.5457245069174027


In [48]:
# after adding det_conf to the map or re-running detector for a small sample
df_small = df.sample(2000, random_state=1)   # or whatever sample
# then inspect det_conf histogram and min_side vs det_conf scatter